# Preprocessing for New and Old Loan Users

This notebook builds a preprocessing pipeline for the final feature sets used in scoring and default prediction. It loads `new_user.csv` and `old_user.csv`, normalizes raw schema aliases, encodes features, and saves processed arrays and pipeline artifacts.

In [1]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler
)

warnings.filterwarnings("ignore")

## Load raw data

Load the source CSV files and inspect the shapes and available columns.

In [2]:
DATA_PATH = Path(
    r"C:\Users\PC\Documents\CADT_Y4\Internship_II\Project\AI_Credit_Scoring_System\ML\data"
)

NEW_PATH = DATA_PATH / "new_user.csv"
OLD_PATH = DATA_PATH / "old_user.csv"

df_new = pd.read_csv(NEW_PATH)
df_old = pd.read_csv(OLD_PATH)

print("New Users:", df_new.shape)
print("Old Users:", df_old.shape)

New Users: (35000, 44)
Old Users: (35000, 49)


In [3]:
SAVE_PATH = Path(
    r"C:\Users\PC\Documents\CADT_Y4\Internship_II\Project\AI_Credit_Scoring_System\ML_Unsupervised\preprocess"
)

SAVE_PATH.mkdir(
    parents=True,
    exist_ok=True
)

## Final feature lists

Define the exact feature schema for both new and old user processing. Missing raw columns are skipped with a warning.

In [4]:
ID_COLUMN = "user_id"

### New user feature list
This list defines the final columns used to preprocess new loan applicants. Missing columns will be skipped with a warning.

In [5]:
NEW_FEATURES = [
    "age",
    "marital_status",
    "dependents",
    "education_level",
    "living_duration",
    "residence_type",
    "is_urban",
    "employment_type",
    "employment_tenure",
    "work_experience",
    "industry",
    "position",
    "monthly_income",
    "monthly_outflow",
    "net_cash_flow",
    "income_source_diversity",
    "expense_ratio",
    "monthly_debt_payment",
    "surplus_income",
    "dti_ratio",
    "lti_ratio",
    "existing_loan_count",
    "total_debt_amount",
    "affordability_ratio",
    "loan_amount",
    "loan_tenure",
    "loan_purpose",
    "guarantee_support",
    "asset_ownership",
    "saving_indicator",
]

### Old user feature list
This list defines the final columns used to preprocess repeat borrowers.

In [6]:
OLD_FEATURES = [
    "loan_cycle_count",
    "age",
    "marital_status",
    "dependents",
    "education_level",
    "living_duration",
    "residence_type",
    "is_urban",
    "employment_type",
    "employment_tenure",
    "work_experience",
    "industry",
    "position",
    "monthly_income",
    "monthly_outflow",
    "net_cash_flow",
    "income_source_diversity",
    "expense_ratio",
    "monthly_debt_payment",
    "surplus_income",
    "dti_ratio",
    "lti_ratio",
    "existing_loan_count",
    "total_debt_amount",
    "affordability_ratio",
    "loan_amount",
    "loan_tenure",
    "loan_purpose",
    "guarantee_support",
    "asset_ownership",
    "on_time_payment_ratio",
    "max_dpd",
    "early_repayment_frequency",
    "saving_indicator"
]

### Feature metadata
Define how ordered and binary fields are encoded, and which columns are treated as numeric.

In [7]:
ORDINAL_FEATURES = {

    "education_level": [
        "High School",
        "Bachelor",
        "Master",
        "PhD"
    ],

    "employment_tenure": [
        "< 3 months",
        "3 - 6 months",
        "6 - 12 months",
        "1 - 3 years",
        "> 3 years"
    ],

    "work_experience": [
        "Less than 1 year",
        "1 to 3 years",
        "3 to 6 years",
        "6 to 9 years",
        "More than 9 years"
    ],
    
    "living_duration": [
        "Less than 3 months",
        "Less than 6 months",
        "Less than 1 year",
        "Less than 2 years",
        "Less than 3 years",
        "More than 3 years"
    ],
    
    "on_time_payment_ratio": [
        "< 50%",
        "50 - 69%",
        "70 - 84%",
        "85 - 94%",
        "≥ 95%"
    ],

    "max_dpd": [
        "0 day",
        "1 - 15 days",
        "> 15 days"
    ],

    "early_repayment_frequency": [
        "> 50%",
        "> 40%",
        "> 30%",
        "> 20%",
        "> 10%",
        "< 10%"
    ],
}

BINARY_FEATURES = {
    "guarantee_support": {
        "No": 0,
        "Yes": 1
    },
    "saving_indicator": {
        "No": 0,
        "Yes": 1
    },
    "is_urban": {
        "No": 0,
        "Yes": 1
    }
}

NUMERIC_FEATURES = [
    "loan_cycle_count",
    "age",
    "dependents",
    "monthly_income",
    "monthly_outflow",
    "net_cash_flow",
    "income_source_diversity",
    "expense_ratio",
    "dti_ratio",
    "lti_ratio",
    "loan_amount",
    "loan_tenure",
    "existing_loan_count",
    "total_debt_amount",
    "affordability_ratio",
    "monthly_debt_payment",
    "surplus_income",
]

NOMINAL_FEATURES = [
    "marital_status",
    "residence_type",
    "employment_type",
    "industry",
    "position",
    "loan_purpose",
    "asset_ownership"
]

## Preprocessing helpers

The next code cell normalizes raw aliases, selects the requested features, fills missing values, scales numeric inputs, and encodes categorical values.

### Normalize and rename raw columns
This cell maps raw column aliases into the final feature names and normalizes the target column when needed.

In [8]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize schema aliases -> final names."""
    alias_map = {
        "industry_type": "industry",
        "on_time_payment_rate": "on_time_payment_ratio",
    }
    df = df.copy()
    for old_col, new_col in alias_map.items():
        if old_col in df.columns and new_col not in df.columns:
            df = df.rename(columns={old_col: new_col})
    return df

### Encode binary fields
Convert Yes/No features into numeric values so the model can learn from them.

In [9]:
def encode_binary(df: pd.DataFrame) -> pd.DataFrame:
    """Map Yes/No-like columns into 0/1 when applicable."""
    df = df.copy()
    for col, mapping in BINARY_FEATURES.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(df[col])
    return df

Feature ENgineering

In [10]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Create net_cash_flow if missing."""
    df = df.copy()
    if (
        "net_cash_flow" not in df.columns
        and "monthly_income" in df.columns
        and "monthly_outflow" in df.columns
    ):
        df["net_cash_flow"] = df["monthly_income"] - df["monthly_outflow"]
    return df

### Build the preprocessing pipeline
Scale numeric features, encode ordinal categories, and one-hot encode remaining categorical fields.

In [11]:
def build_preprocessor(feature_cols):
    """Create ColumnTransformer for numeric/ordinal/nominal."""
    numeric_cols = [c for c in feature_cols if c in NUMERIC_FEATURES]
    ordinal_cols = [c for c in feature_cols if c in ORDINAL_FEATURES]
    binary_cols = [c for c in feature_cols if c in BINARY_FEATURES]
    nominal_cols = [c for c in feature_cols if c not in numeric_cols + ordinal_cols + binary_cols]

    numeric_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]
    )

    ordinal_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OrdinalEncoder(
                    categories=[ORDINAL_FEATURES[c] for c in ordinal_cols],
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                ),
            ),
        ]
    )

    nominal_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )

    preprocessor = ColumnTransformer(
        [
            ("num", numeric_pipeline, numeric_cols),
            ("ord", ordinal_pipeline, ordinal_cols),
            ("bin", "passthrough", binary_cols),
            ("nom", nominal_pipeline, nominal_cols),
        ]
    )
    return preprocessor

In [12]:
def split_train_test(X):

    X_train, X_test = train_test_split(
        X,
        test_size=0.20,
        random_state=42
    )

    return X_train, X_test

In [13]:
def save_matrix(X, filepath, feature_names=None):

    if sparse.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X)

    if feature_names is not None:
        df.columns = feature_names

    df.to_csv(
        str(filepath) + ".csv",
        index=False
    )

In [16]:
def preprocess_unsupervised_dataset(df: pd.DataFrame, feature_list, prefix: str):
    print(f"\nProcessing {prefix}...")

    df = normalize_columns(df)
    df = feature_engineering(df)
    df = encode_binary(df)

    # Keep only columns we need (plus target + id if present)
    needed = set(feature_list) | {ID_COLUMN}
    available_cols = [c for c in df.columns if c in needed]
    df = df[available_cols].copy()

    # X = drop target + id
    X = df.drop(columns=[ID_COLUMN], errors="ignore")

    # Split
    X_train, X_test = split_train_test(X)

    # Build & fit preprocessor
    preprocessor = build_preprocessor(list(X_train.columns))

    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p = preprocessor.transform(X_test)

    # Feature names
    feature_names = preprocessor.get_feature_names_out()

    # Save X
    save_matrix(X_train_p, SAVE_PATH / f"{prefix}_X_train", feature_names=feature_names)
    save_matrix(X_test_p, SAVE_PATH / f"{prefix}_X_test", feature_names=feature_names)

    # Save preprocessor
    joblib.dump(preprocessor, SAVE_PATH / f"{prefix}_preprocessor_unsupervised.pkl")

    # Save feature names list
    pd.Series(feature_names).to_csv(SAVE_PATH / f"{prefix}_feature_names.csv", index=False)

    # Validate saved output shapes
    for part in ["X_train", "X_test"]:
        path = SAVE_PATH / f"{prefix}_{part}.csv"
        chk = pd.read_csv(path)
        print(f"Saved {path.name}: {chk.shape}")

    print(prefix, "completed.")
    print("Transformed shapes:", X_train_p.shape, X_test_p.shape)


In [17]:
preprocess_unsupervised_dataset(df_new, NEW_FEATURES, "new")
preprocess_unsupervised_dataset(df_old, OLD_FEATURES, "old")


Processing new...
Saved new_X_train.csv: (28000, 63)
Saved new_X_test.csv: (7000, 63)
new completed.
Transformed shapes: (28000, 63) (7000, 63)

Processing old...
Saved old_X_train.csv: (28000, 67)
Saved old_X_test.csv: (7000, 67)
old completed.
Transformed shapes: (28000, 67) (7000, 67)
